# RFZ 규제자유특구사례 크롤러 (수정판)
- 각 탭을 `active → 파싱 → 다음` 순으로 순회
- 컬럼: **대제목, 중제목, 소제목, 현황, 허용**
- 최종 값에 **특수문자 제거 정규식** 적용 (한글/영문/숫자/공백만 유지)
- 아래 셀을 실행하세요. 필요 시 `headless=False`로 변경

In [38]:
# RFZ 규제샌드박스 크롤러 (중제목 중복/누락 수정: visible-only + empty mid skip)
# - 표시된(visible) 컨테이너/헤더/ul만 대상으로 파싱
# - h4 바로 다음 "표시된" ul[1] 묶음만 사용
# - 중제목이 비면 행 생성 스킵 (빈 중제목 행 제거)
# - '☞ … 허용' 문장 → 허용 컬럼
# - CSV QUOTE_ALL (콤마 안전)

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from bs4 import BeautifulSoup
import pandas as pd
import re, time, csv

URL = "https://rfz.go.kr/?menuno=206#none"

# ---------- text helpers ----------
def normalize_spaces(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "").strip())

def clean_major_title(s: str) -> str:
    s = normalize_spaces(s)
    # "10차 - " / "10차" / "①차 -" 제거
    s = re.sub(r'^\s*(?:\d+|[①-⑳])\s*차\s*[-–—·\.\)]*\s*', '', s)
    return s

def clean_mid_title(s: str) -> str:
    s0 = normalize_spaces(s)
    # "실증특례" 접두 제거
    s1 = re.sub(
        r'^\s*[\u2460-\u2473]?\s*(?:\(|\[)?\s*실증\s*특례\s*(?:\)|\])?\s*[-–—:·\.\)]*\s*',
        '',
        s0
    )
    return s1 if s1 else s0

def first_sentence(txt: str, max_len=120) -> str:
    txt = normalize_spaces(txt)
    m = re.search(r'(.+?)(?:[.!?。…]|$)', txt)
    cand = m.group(1) if m else txt
    return cand[:max_len].strip()

def extract_subtitle_from_block(html: str) -> str:
    soup = BeautifulSoup(html or "", "html.parser")
    st = soup.find("strong")
    if st:
        return normalize_spaces(st.get_text(" ", strip=True))
    flat = soup.get_text(" ", strip=True)
    m = re.search(r'\[([^\]]+)\]', flat)
    if m:
        return normalize_spaces(m.group(1))
    flat2 = re.sub(r'(?:\(|\[)?\s*(현황|허용|조건부\s*허용)\s*(?:\)|\])', ' ||LABEL|| ', flat)
    pre = flat2.split('||LABEL||')[0]
    return first_sentence(pre) if pre.strip() else ""

def strip_label_prefix(s: str, label_regex: str) -> str:
    return re.sub(label_regex, "", s, flags=re.I).strip()

# ---------- 현황/허용 파서 ----------
def parse_stat_allow_from_block_html(html: str):
    soup = BeautifulSoup(html or "", "html.parser")
    parts_stat, parts_allow = [], []
    state = None

    label_pat = re.compile(
        r'^\s*(?:\(|\[)?\s*(현황|허용|조건부\s*허용)\s*(?:\)|\])?\s*[:\-–—·\.\)]*\s*',
        re.I
    )

    elems = soup.find_all(['p', 'div'])
    for el in elems:
        txt = el.get_text(" ", strip=True)
        if not txt:
            continue

        label_hit = None
        for b in el.find_all(['b', 'strong']):
            t = b.get_text(" ", strip=True)
            if re.search(r'현황', t):
                label_hit = 'stat'; break
            if re.search(r'(?:조건부\s*)?허용', t):
                label_hit = 'allow'; break

        m = label_pat.match(txt)
        if m:
            key = m.group(1)
            state = 'allow' if re.search(r'허용', key) else 'stat'
            txt = txt[m.end():].strip()
        elif label_hit:
            state = label_hit
            txt = strip_label_prefix(
                txt,
                r'^\s*(?:\(|\[)?\s*(?:현황|허용|조건부\s*허용)\s*(?:\)|\])?\s*[:\-–—·\.\)]*\s*'
            )
        else:
            # '☞' 시작 + '허용' 포함 → 허용
            if txt.lstrip().startswith('☞') and ('허용' in txt):
                state = 'allow'
                txt = txt.lstrip('☞').strip()

        if state == 'stat':
            parts_stat.append(txt)
        elif state == 'allow':
            parts_allow.append(txt)

    stat = normalize_spaces(" ".join(parts_stat))
    allow = normalize_spaces(" ".join(parts_allow))

    if not stat and not allow:
        flat = soup.get_text(" ", strip=True)
        t = re.sub(r'[\(\[]\s*현황\s*[\)\]]', ' ||현황|| ', flat)
        t = re.sub(r'[\(\[]\s*(?:조건부\s*)?허용\s*[\)\]]', ' ||허용|| ', t)
        t = re.sub(r'(?<!\|)\b현황\b(?!\|)', ' ||현황|| ', t)
        t = re.sub(r'(?<!\|)\b(?:조건부\s*)?허용\b(?!\|)', ' ||허용|| ', t)
        t = re.sub(r'☞[^|]*허용[^|]*', lambda m: ' ||허용|| ' + m.group(0).lstrip('☞').strip() + ' ', t)
        m1 = re.search(r'\|\|현황\|\|\s*(.*?)\s*(?=\|\|허용\|\||$)', t, re.S)
        m2 = re.search(r'\|\|허용\|\|\s*(.*)$', t, re.S)
        if m1: stat  = normalize_spaces(m1.group(1))
        if m2: allow = normalize_spaces(m2.group(1))

    stat  = strip_label_prefix(stat,  r'^\s*(?:\(|\[)?\s*현황\s*(?:\)|\])?\s*[:\-–—·\.\)]*\s*')
    allow = strip_label_prefix(allow, r'^\s*(?:\(|\[)?\s*(?:조건부\s*)?허용\s*(?:\)|\])?\s*[:\-–—·\.\)]*\s*')
    return stat, allow

# ---------- selenium helpers ----------
def build_driver(headless=True):
    opts = webdriver.ChromeOptions()
    if headless:
        opts.add_argument("--headless=new")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    driver = webdriver.Chrome(options=opts)
    driver.set_window_size(1400, 2000)
    return driver

def wait_active(driver, li_el, timeout=10):
    WebDriverWait(driver, timeout).until(
        lambda d: "active" in (li_el.get_attribute("class") or "").lower()
    )

def click_anchor(driver, li_el):
    a = li_el.find_element(By.XPATH, ".//a")
    driver.execute_script("arguments[0].click();", a)

def visible_only(elems):
    return [e for e in elems if e.is_displayed()]

# ---------- parse one tab (<li>) ----------
def parse_li(li_el):
    rows = []
    major = clean_major_title(li_el.find_element(By.XPATH, ".//a").text)

    # 표시된 템플릿 스코프만 선택 (after/before 중 실제로 보이는 쪽)
    scopes = li_el.find_elements(
        By.XPATH,
        ".//div[contains(@class,'con-template-list') or contains(@class,'before')]"
    )
    scopes = visible_only(scopes)
    scope = scopes[0] if scopes else li_el  # 그래도 없으면 li 전체에서 검색

    # 표시된 h4만 순회
    h4s = scope.find_elements(By.XPATH, ".//h4")
    h4s = visible_only(h4s)
    if not h4s:
        return rows

    for h4 in h4s:
        mid_raw = h4.text
        mid = clean_mid_title(mid_raw).strip()
        if not mid:  # 빈 중제목 행 생성 방지
            continue

        # h4의 '표시된' 다음 형제 ul[1]
        uls = h4.find_elements(By.XPATH, "following-sibling::ul[1]")
        uls = visible_only(uls)
        if not uls:
            # 부모 기준 보조 탐색
            try:
                parent = h4.find_element(By.XPATH, "..")
                uls = visible_only(parent.find_elements(By.XPATH, "./ul[1]"))
            except:
                uls = []
        if not uls:
            continue

        ul = uls[0]
        li_items = ul.find_elements(By.XPATH, "./li")
        li_items = visible_only(li_items)

        for it in li_items:
            blocks = it.find_elements(
                By.XPATH,
                ".//div[contains(@class,'txt-area') or contains(@class,'text')]"
            )
            blocks = visible_only(blocks)
            if not blocks:
                blocks = [it]  # 최후수단

            for b in blocks:
                html = b.get_attribute("innerHTML") or ""
                sub = extract_subtitle_from_block(html)
                stat, allow = parse_stat_allow_from_block_html(html)
                rows.append([major, mid, sub, stat, allow])

    return rows

# ---------- main ----------
def crawl_rfz(headless=True, out_csv="rfz_crawl.csv", dedup=True):
    driver = build_driver(headless=headless)
    all_rows = []
    try:
        driver.get(URL)
        WebDriverWait(driver, 12).until(EC.presence_of_element_located((By.CSS_SELECTOR, "form > ul > li")))
        lis = driver.find_elements(By.CSS_SELECTOR, "form > ul > li")

        for li in lis:
            click_anchor(driver, li)
            try:
                wait_active(driver, li, 10)
            except TimeoutException:
                click_anchor(driver, li)
                wait_active(driver, li, 10)

            all_rows.extend(parse_li(li))
            time.sleep(0.1)
    finally:
        try:
            driver.quit()
        except:
            pass

    df = pd.DataFrame(all_rows, columns=["대제목","중제목","소제목","현황","허용"])

    if dedup and not df.empty:
        df = df.drop_duplicates(
            subset=["대제목","중제목","소제목","현황","허용"],
            keep="first"
        ).reset_index(drop=True)

    df.to_csv(
        out_csv,
        index=False,
        encoding="utf-8-sig",
        quoting=csv.QUOTE_ALL,
        quotechar='"',
        doublequote=True,
        lineterminator="\n"
    )
    print(f"Saved to {out_csv}")
    return df

# 사용 예:
# df = crawl_rfz(headless=True, out_csv="rfz_crawl.csv")
# df.head(10)


In [39]:
df = crawl_rfz(headless=True)

Saved to rfz_crawl.csv


In [35]:
df

,대제목,중제목,소제목,현황,허용
0,울산 암모니아 벙커링 규제자유특구 지정,중대형 선박용 Truck To Ship 벙커링 안전기술개발 및 실증,[암모니아 Truck to Ship 벙커링 특례],암모니아 연료선박 상용화 초기단계에서는 가장 현실적이고 안정성을 신속하게 검증할 수...,암모니아 연료선박 대상 Truck to Ship(탱크로리→선박 직접 공급) 벙커링 ...
1,울산 암모니아 벙커링 규제자유특구 지정,,[암모니아 Truck to Ship 벙커링 특례],암모니아 연료선박 상용화 초기단계에서는 가장 현실적이고 안정성을 신속하게 검증할 수...,암모니아 연료선박 대상 Truck to Ship(탱크로리→선박 직접 공급) 벙커링 ...
2,대전 우주기술 연구·활용 규제자유특구,우주추진용 고압가스 부품 기술기준 정립 및 시험·실증,[우주추진용 고압가스 부품 기준 완화 특례],"발사체 및 위성 등 우주 추진용 고압가스부품(용기, 밸브 등)에 대한 안전기준 및 ...",우주 추진용 고압가스부품에 대한 안전기준 및 시험방법 등 기준 마련을 위한 실증 허용
3,대전 우주기술 연구·활용 규제자유특구,,[우주추진용 고압가스 부품 기준 완화 특례],"발사체 및 위성 등 우주 추진용 고압가스부품(용기, 밸브 등)에 대한 안전기준 및 ...",우주 추진용 고압가스부품에 대한 안전기준 및 시험방법 등 기준 마련을 위한 실증 허용
4,전북 기능성식품 규제자유특구 지정,미등재 고시형 기능성 원료의 일반식품 적용 기준 및 규격 실증,[일반식품 사용 가능 원료 확대],기존 건강기능식품 고시형 기능성 원료 68종 중 29종에 한해 일반식품 사용 허용(...,관계부처와 협의한 기능성 원료에 한해 실증 허용
...,...,...,...,...,...
311,부산 블록체인 규제자유특구 지정,,[위치정보시스템 특례],"블록체인 기반 영상 제보 및 공유 시스템 활용 예정이나, 위치정보사업자는 위치정보를...",
312,부산 블록체인 규제자유특구 지정,,[개인정보 및 개인위치정보 파기의무 특례],"❶) 개인위치정보 삭제 의무 존재하나(위치정보법), 블록체인 기술 특성상 등록된 정...",개인위치정보 삭제의무에 대해 오프체인(Off-Chain) 저장·파기 방식의 특례 허용
313,부산 블록체인 규제자유특구 지정,,"(현황❷) 개인정보 처리목적 달성 후 파기 의무 존재(개인정보보호법), 블록체인 기...","❷) 개인정보 처리목적 달성 후 파기 의무 존재(개인정보보호법), 블록체인 기술 특...",개인정보 파기의무에 대해 오프체인(Off-Chain) 저장·파기 방식의 특례 허용
314,부산 블록체인 규제자유특구 지정,,[선불수단 양도 특례],선불전자지급수단 양도시 반드시 발행자의 중앙전산시스템을 경유하여야하나(전자금융거래법...,분산원장상 합의로 선불수단 양도 인정 특례 허용
